# Top-ranked units (Gerken, Darcher et al. 2025, dandiset 001616)

Paper: Gerken, Darcher et al. (2025), eLife reviewed preprint 106758 v1,
[doi:10.7554/eLife.106758.1](https://doi.org/10.7554/eLife.106758.1), full text.
Dandiset: [DANDI:001616](https://dandiarchive.org/dandiset/001616/0.260702.0824), version 0.260702.0824.

Companion to `reproduce_gerken_2025.ipynb`, with the same loading, samples, folds and logistic
regression (`summer.py`). It holds two claims:

| Claim | Status |
|---|---|
| fig7b-top500-overlap | tested |
| fig7a-500-neurons | tested, with the logistic regression in place of the LSTM |

The notebook takes about 9 minutes.

In [ ]:
import os
for var in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(var, "1")
import time

import numpy as np
import pandas as pd

import summer

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)

started_all = started = time.time()
sessions = summer.load_all()
units = summer.unit_table(sessions)
spikes = summer.movie_spikes(sessions)
print(f"loaded {len(sessions)} files, {len(units)} units, from dandiset {summer.DANDISET_ID} "
      f"version {summer.DANDISET_VERSION} in {time.time() - started:.0f} s")

In [ ]:
edges = sessions[0]["edges_s"]
S = summer.sample_windows(spikes, edges)
n = S.shape[0]
y = summer.bin_labels(sessions[0], "summer")
print(f"patients {units.patient.nunique()}, units {len(units)}; by region {units.region.value_counts().to_dict()}")
print(f"samples x units: {S.shape}; Summer positive in {np.mean(y == 1):.3f} of the samples, values {sorted(np.unique(y).tolist())}")
for f in range(5):
    print(f"test fold {f + 1}: " + "; ".join(f"validation fold {v + 1} train {len(a)}, validation {len(b)}, test {len(c)}"
                                            for v in range(5) if v != f for a, b, c in [summer.split(n, f, val_fold=v)]))

## fig7b-top500-overlap

> Across the five rankings of neurons by logistic regression weights for Summer, the top 500
> neurons share 78 neurons, against about 1 expected by chance.

In [ ]:
rng_rank = np.random.default_rng(37)
started = time.time()
keys, jobs = [], []
for f in range(5):
    for v in [j for j in range(5) if j != f]:
        keys.append((f, v))
        jobs.append((summer.fit_logreg, (S, y, *summer.split(n, f, val_fold=v))))
kfits = dict(zip(keys, summer.run_jobs(jobs)))
print(f"{len(jobs)} sub-split fits in {time.time() - started:.0f} s")
rankings, zero_in_top = [], []
for f in range(5):
    coefs = [kfits[(f, v)]["coef"] for v in range(5) if v != f]
    order = summer.ranking(coefs, rng_rank)
    rankings.append(order)
    nz = (np.abs(np.array(coefs)) > 0).any(0)
    zero_in_top.append(int((~nz[order[:500]]).sum()))
    print(f"test fold {f + 1}: C {[kfits[(f, v)]['C'] for v in range(5) if v != f]}, units with a non-zero coefficient in any of the 4 models "
          f"{int(nz.sum())}, in all 4 {int((np.abs(np.array(coefs)) > 0).all(0).sum())}; top 500 tied at zero {zero_in_top[-1]}; "
          f"sub-split test kappa {np.round([kfits[(f, v)]['kappa'] for v in range(5) if v != f], 3).tolist()}")
N = S.shape[1]
rows = []
for k_ in (100, 350, 500, 750, 1000):
    common = set(rankings[0][:k_])
    for o in rankings[1:]:
        common &= set(o[:k_])
    sims = [len(set.intersection(*[set(rng_rank.choice(N, k_, replace=False)) for _ in range(5)])) for _ in range(200)]
    rows.append({"top": k_, "common": len(common), "chance_formula": N * (k_ / N) ** 5, "chance_simulated": np.mean(sims),
                 "paper_chance": {100: 0, 350: 0, 500: 1, 750: 9, 1000: 37}[k_]})
ov = pd.DataFrame(rows)
print(ov.round(2).to_string(index=False))
common500 = set(rankings[0][:500]).intersection(*[set(o[:500]) for o in rankings[1:]])
print("common 500 by region:", units.loc[sorted(common500)].region.value_counts().to_dict(),
      "; patients:", units.loc[sorted(common500)].patient.nunique())
# Variant: zero-tied units ordered by unit index instead of a random draw (same order in every ranking).
order_idx = [np.lexsort((np.arange(N), -np.abs(np.array([kfits[(f, v)]["coef"] for v in range(5) if v != f])).mean(0),
                         -(np.abs(np.array([kfits[(f, v)]["coef"] for v in range(5) if v != f])) > 0).sum(0))) for f in range(5)]
print("variant, ties by unit index: common in top 500:", len(set(order_idx[0][:500]).intersection(*[set(o[:500]) for o in order_idx[1:]])))

**Result: partially reproduced.** The five rankings share 95 units in their top 500, against 78 in
the paper (tolerance 70.2 to 85.8) and 1.14 expected by chance (1.12 simulated), 83 times chance.
No unit in a top 500 was tied at zero, so the tie-break does not matter (95 with ties by unit
index). The overlaps for the top 100, 350, 750 and 1000 are 20, 65, 156 and 252, against chance
values of 0.00, 0.19, 8.69 and 36.6 that match the paper's (0, 0, 9, 37). The 95 units come from 26
patients and every region.

## fig7a-500-neurons

> Decoding Summer from the 500 top-ranked neurons gives the peak performance (kappa 0.38), above
> that of the full population.

For each test fold, the logistic regression is refitted on each of its four sub-splits with only
the top k units of that fold's ranking, and scored on the shared test fold. Kappa is averaged over
the four sub-splits, then over the five test folds (Methods); the SEM is over the five fold means.

In [ ]:
rng_top = np.random.default_rng(39)
SIZES = (23, 114, 150, 229, 500, 750, 1000, 1500)
started = time.time()
keys = [(k_, f, v) for k_ in SIZES for f in range(5) for v in range(5) if v != f]
tfits = dict(zip(keys, summer.run_jobs([(summer.fit_logreg, (S, y, *summer.split(n, f, val_fold=v), rankings[f][:k_]))
                                         for k_, f, v in keys])))
tfits |= {(N, f, v): r for (f, v), r in kfits.items()}
print(f"{len(keys)} fits on top-ranked units in {time.time() - started:.0f} s")
rows = []
for k_ in SIZES + (N,):
    per_fold = np.array([np.mean([tfits[(k_, f, v)]["kappa"] for v in range(5) if v != f]) for f in range(5)])
    pmax = max(summer.permutation_p(tfits[(k_, f, v)], rng_top) for f in range(5) for v in range(5) if v != f)
    rows.append({"units": k_, "percent": round(100 * k_ / N, 1), "kappa": per_fold.mean(), "sem": per_fold.std(ddof=1) / np.sqrt(5),
                 "fold_means": np.round(per_fold, 3).tolist(), "max_perm_p": pmax})
top = pd.DataFrame(rows).set_index("units")
print(top.round(4).to_string())
print("paper (LSTM): peak 0.38 at 500 units; 0.52 at 150 units with a ranking that saw the test data")
best = top.kappa.idxmax()
print(f"500 units {top.kappa[500]:.4f}; all {N} units {top.kappa[N]:.4f}; difference {top.kappa[500] - top.kappa[N]:.4f}; "
      f"highest at {best} units ({top.kappa[best]:.4f})")

**Result: partially reproduced.** With the logistic regression, the top 500 units decode Summer
with kappa 0.375, above all 2286 units (0.357), by 0.018, less than the SEM over folds (0.026 and
0.029). The peak is not at 500: 229 units give 0.394 and 150 units 0.389; from 229 units upwards the
kappa falls steadily, and the top 23 units still give 0.341. Every fit is above chance by the
test-label shuffle (p = 0.001). The paper's LSTM peaks at 0.38 with 500 units.

In [ ]:
print(f"total run time {(time.time() - started_all) / 60:.1f} min")